## Setup

First, let's import the necessary modules and verify our environment.

In [1]:
import numpy as np
import tgraph.transform as tf

# Configure numpy printing
np.set_printoptions(precision=3, suppress=True)

print("✓ tgraph imported successfully")

✓ tgraph imported successfully


## 1. Basic Transforms

### Creating Transforms

The core of tgraph is the `Transform` class, which represents SE(3) rigid body transformations (rotation + translation).

### Euler Angle Convention (Roll-Pitch-Yaw)

tgraph uses the **aerospace/robotics convention** for Euler angles:

| Angle | Axis | Description |
|-------|------|-------------|
| **Roll (φ)** | X-axis | Banking left/right |
| **Pitch (θ)** | Y-axis | Nose up/down |
| **Yaw (ψ)** | Z-axis | Heading direction |

Rotations are applied in **intrinsic ZYX order**: yaw → pitch → roll.

This convention is standard in:
- ✈️ Aerospace (heading, pitch, bank)
- 🤖 Robotics (ROS REP-103)
- 🚢 Marine vehicles

All angles are in **radians**.

In [ ]:
# Create a simple translation
translation_only = tf.Translation(x=1.0, y=2.0, z=3.0)
print("Translation:")
print(translation_only)
print("\nAs 4x4 matrix:")
print(translation_only.as_matrix())

Translation:
Transform(translation=[1.000, 2.000, 3.000], rotation=quaternion(1, 0, 0, 0))

As 4x4 matrix:
[[1. 0. 0. 1.]
 [0. 1. 0. 2.]
 [0. 0. 1. 3.]
 [0. 0. 0. 1.]]


: 

: 

In [ ]:
# Create a rotation around Z-axis (90 degrees) using roll-pitch-yaw
rotation_only = tf.from_euler_angles(yaw=np.pi / 2)

print("Rotation (90° yaw):")
print(rotation_only)
print("\nAs 4x4 matrix:")
print(rotation_only.as_matrix())

# Extract Euler angles back
roll, pitch, yaw = rotation_only.as_euler_angles()
print(f"\nEuler angles: roll={np.degrees(roll):.1f}°, pitch={np.degrees(pitch):.1f}°, yaw={np.degrees(yaw):.1f}°")

AttributeError: module 'tgraph.transform' has no attribute 'from_euler_angles'

: 

: 

In [ ]:
# Create a full SE(3) transform (rotation + translation)
full_transform = tf.Transform(
    translation=[1.0, 0.0, 0.5],
    rotation=tf.Rotation.from_euler_angles(yaw=np.pi / 4).rotation  # 45° heading
)
print("Full Transform (45° yaw + translation):")
print(full_transform)
print("\nAs 4x4 matrix:")
print(full_transform.as_matrix())

: 

: 

### Applying Transforms to Points

Transforms can be applied to 3D points to move them between coordinate frames.

In [ ]:
# Define some points in 3D space
points = np.array([
    [1.0, 0.0, 0.0],  # Point on X-axis
    [0.0, 1.0, 0.0],  # Point on Y-axis
    [0.0, 0.0, 1.0],  # Point on Z-axis
])

print("Original points:")
print(points)

# Apply the 90° Z-rotation
rotated_points = rotation_only.apply(points)
print("\nAfter 90° Z-rotation:")
print(rotated_points)
print("Note: X-axis point → Y-axis, Y-axis point → -X-axis")

: 

: 

## 2. Transform Composition

Transforms can be composed using the `*` operator. The convention is: `(T1 * T2) * p = T1 * (T2 * p)`

In [ ]:
# Create two transforms
translate_x = tf.Translation(x=2.0)
rotate_z = tf.Rotation.from_euler_angles(yaw=np.pi / 2)

# Compose them: first rotate, then translate
composed = translate_x * rotate_z

# Test with a point
point = np.array([[1.0, 0.0, 0.0]])

print("Original point:", point[0])
print("After rotation:", rotate_z.apply(point)[0])
print("After rotation + translation:", composed.apply(point)[0])

# Verify composition
manual_result = translate_x.apply(rotate_z.apply(point))
print("Manual composition:", manual_result[0])
print("Match:", np.allclose(composed.apply(point), manual_result))

: 

: 

### Transform Inversion

Every SE(3) transform is invertible. The inverse operation reverses the transformation.

In [ ]:
# Create a transform with roll-pitch-yaw rotation
original = tf.Transform(
    translation=[3.0, 4.0, 5.0],
    rotation=tf.Rotation.from_euler_angles(roll=0.1, pitch=0.2, yaw=0.3).rotation
)

# Get its inverse
inverse = original.inverse()

# Test: applying both should give identity
should_be_identity = original * inverse
identity_matrix = should_be_identity.as_matrix()

print("Original * Inverse (should be identity):")
print(identity_matrix)
print("\nIs identity?", np.allclose(identity_matrix, np.eye(4)))

: 

: 

## 3. Transform Graphs

The `TransformGraph` manages relationships between multiple coordinate frames, automatically computing paths and caching results.

In [ ]:
# Create a transform graph
graph = tf.TransformGraph()

# Build a simple robot: world → base → arm → gripper
graph.add_transform('world', 'base', tf.Translation(x=0.0, y=0.0, z=0.5))
graph.add_transform('base', 'arm', tf.Transform(
    translation=[0.2, 0.0, 0.3],
    rotation=tf.Rotation.from_euler_angles(pitch=np.pi / 6).rotation  # 30° pitch
))
graph.add_transform('arm', 'gripper', tf.Translation(x=0.4, y=0.0, z=0.0))

print("Frames in graph:", graph.list_frames())

: 

: 

In [ ]:
# Query transform between any two frames
world_to_gripper = graph.get_transform('world', 'gripper')
print("Transform from world to gripper:")
print(world_to_gripper.as_matrix())

# The graph automatically inverts when needed
gripper_to_world = graph.get_transform('gripper', 'world')
print("\nTransform from gripper to world (inverse):")
print(gripper_to_world.as_matrix())

: 

: 

In [ ]:
# Transform points between frames
point_in_gripper = np.array([[0.05, 0.0, 0.0]])  # 5cm in front of gripper

point_in_world = world_to_gripper.apply(point_in_gripper)
print("Point in gripper frame:", point_in_gripper[0])
print("Same point in world frame:", point_in_world[0])

: 

: 

### Graph Caching & Performance

The graph caches computed transforms as "shortcuts" for faster lookup.

In [ ]:
# First query computes the path
print("Before caching - number of edges:", graph.graph.number_of_edges())

# This query triggers caching
_ = graph.get_transform('world', 'gripper')

print("After caching - number of edges:", graph.graph.number_of_edges())
print("(Shortcut edge added for faster future lookups)")

# View all edges
print("\nEdges in graph:")
for u, v, data in graph.graph.edges(data=True):
    edge_type = "Cache" if data.get('is_cache', False) else "Ground Truth"
    print(f"  {u} → {v} ({edge_type})")

: 

: 

## 4. Camera Projections

tgraph supports 3D → 2D projections for camera models.

In [ ]:
# Create a camera with known intrinsics
focal_length = 500.0
principal_point = (320.0, 240.0)  # Image center

K = np.array([
    [focal_length, 0, principal_point[0]],
    [0, focal_length, principal_point[1]],
    [0, 0, 1]
])

# Camera pose (looking down the Z-axis)
R = np.eye(3)  # No rotation
t = np.array([[0], [0], [0]])  # At origin

camera = tf.CameraProjection(K=K, R=R, t=t)
print("Camera focal length:", camera.focal_length)
print("Principal point:", camera.principal_point)

: 

: 

In [ ]:
# Project 3D points to 2D image coordinates
points_3d = np.array([
    [0.0, 0.0, 1.0],   # Point at Z=1m, center
    [0.1, 0.0, 1.0],   # Shifted right
    [0.0, 0.1, 1.0],   # Shifted up
    [0.0, 0.0, 2.0],   # Farther away (should project to same pixel)
])

pixels = camera.apply(points_3d)
print("3D Points:")
print(points_3d)
print("\n2D Pixels (u, v):")
print(pixels)
print("\nNote: Point at (0,0,1) projects to principal point (320, 240)")

: 

: 

### Unprojection (2D → 3D)

To recover 3D points from pixels, we need depth information.

In [ ]:
# Get the inverse projection
inv_camera = camera.inverse()

# Unproject pixels back to 3D (requires depth)
pixels_2d = np.array([
    [320.0, 240.0],  # Center pixel
    [420.0, 240.0],  # 100px right
])
depths = np.array([1.0, 1.5])  # Depths in meters

points_3d_recovered = inv_camera.unproject(pixels_2d, depths)
print("Pixels:", pixels_2d)
print("Depths:", depths)
print("Recovered 3D points:")
print(points_3d_recovered)

# Verify by projecting back
pixels_check = camera.apply(points_3d_recovered)
print("\nRe-projected pixels (should match original):")
print(pixels_check)

: 

: 

## 5. Serialization

All transforms can be serialized to/from dictionaries for storage or network transfer.

In [ ]:
# Create various transforms
transforms = [
    tf.Transform(translation=[1, 2, 3], rotation=tf.Rotation.from_euler_angles(yaw=0.5).rotation),
    tf.Translation(x=5.0),
    tf.Rotation.from_euler_angles(roll=0.1, pitch=0.2, yaw=0.3),
    camera,
    tf.Identity()
]

# Serialize each one
serialized = [tf.serialize_transform(t) for t in transforms]

print("Serialized Transform:")
print(serialized[0])
print("\nSerialized CameraProjection:")
print(serialized[3])

: 

: 

In [ ]:
# Deserialize back to objects
deserialized = [tf.deserialize_transform(d) for d in serialized]

# Verify they match
for original, restored in zip(transforms, deserialized):
    match = np.allclose(original.as_matrix(), restored.as_matrix())
    print(f"{type(original).__name__:20} → {type(restored).__name__:20} : {match}")

: 

: 

### Graph Serialization

Entire transform graphs can be saved and loaded.

In [ ]:
# Serialize the robot graph
graph_data = graph.to_dict()

print("Serialized graph keys:", graph_data.keys())
print("Number of nodes:", len(graph_data['nodes']))
print("Number of links:", len(graph_data['links']))
print("\nFirst link:")
print(graph_data['links'][0])

: 

: 

In [ ]:
# Restore from serialized data
restored_graph = tf.TransformGraph.from_dict(graph_data)

# Verify it works
original_tf = graph.get_transform('world', 'gripper')
restored_tf = restored_graph.get_transform('world', 'gripper')

print("Original and restored graphs match:", 
      np.allclose(original_tf.as_matrix(), restored_tf.as_matrix()))

: 

: 

## 6. Practical Example: Multi-Camera Robot

Let's build a more complex example: a mobile robot with two cameras.

In [ ]:
# Create a comprehensive robot setup
robot_graph = tf.TransformGraph()

# World frame is our reference
# Robot base is on the ground, 1m forward from world origin
robot_graph.add_transform('world', 'robot_base', 
                         tf.Translation(x=1.0, y=0.0, z=0.0))

# Camera mast is 0.5m above base
robot_graph.add_transform('robot_base', 'camera_mast',
                         tf.Translation(x=0.0, y=0.0, z=0.5))

# Left camera: 10cm left, looking forward
robot_graph.add_transform('camera_mast', 'left_camera',
                         tf.Translation(x=0.0, y=0.1, z=0.0))

# Right camera: 10cm right, looking forward
robot_graph.add_transform('camera_mast', 'right_camera',
                         tf.Translation(x=0.0, y=-0.1, z=0.0))

# Add camera projections
K_cam = np.array([[400, 0, 320], [0, 400, 240], [0, 0, 1]])
left_proj = tf.CameraProjection(K=K_cam, R=np.eye(3), t=np.zeros((3, 1)))
right_proj = tf.CameraProjection(K=K_cam, R=np.eye(3), t=np.zeros((3, 1)))

robot_graph.add_transform('left_camera', 'left_image', left_proj)
robot_graph.add_transform('right_camera', 'right_image', right_proj)

print("Robot setup complete!")
print("Frames:", robot_graph.list_frames())

: 

: 

In [ ]:
# Simulate observing a point in the world
world_point = np.array([[1.5, 0.0, 0.3]])  # 50cm in front, 30cm high

# Transform to left camera frame
world_to_left = robot_graph.get_transform('world', 'left_camera')
point_in_left_cam = world_to_left.apply(world_point)

# Transform to right camera frame
world_to_right = robot_graph.get_transform('world', 'right_camera')
point_in_right_cam = world_to_right.apply(world_point)

print("Point in world:", world_point[0])
print("Point in left camera:", point_in_left_cam[0])
print("Point in right camera:", point_in_right_cam[0])
print("\nBaseline (difference in Y):", 
      point_in_left_cam[0, 1] - point_in_right_cam[0, 1])

: 

: 

In [ ]:
# Project to both camera images
world_to_left_image = robot_graph.get_transform('world', 'left_image')
world_to_right_image = robot_graph.get_transform('world', 'right_image')

left_pixel = world_to_left_image.apply(world_point)
right_pixel = world_to_right_image.apply(world_point)

print("Left camera pixel:", left_pixel[0])
print("Right camera pixel:", right_pixel[0])
print("\nDisparity (difference in u):", left_pixel[0, 0] - right_pixel[0, 0])
print("(This could be used for stereo depth estimation)")

: 

: 

## Summary

You've learned the core features of `tgraph`:

✓ **Basic transforms**: SE(3) rigid body transforms with quaternions  
✓ **Composition & Inversion**: Chain transforms and compute inverses  
✓ **Transform Graphs**: Manage complex frame hierarchies with automatic path finding  
✓ **Camera Projections**: 3D→2D projection and 2D→3D unprojection with depth  
✓ **Serialization**: Save/load transforms and graphs to/from JSON  
✓ **Performance**: Automatic caching for fast repeated queries  

## Next Steps

- Explore the API documentation
- Try visualizing your graphs (when `tgraph.visualization` is available)
- Build your own robot kinematic models
- Implement SLAM or multi-view geometry applications

Happy transforming! 🚀